# LLM Vector Translator — Reproducible Notebook

This notebook reproduces the key results of the LLM Vector Translator project:

- **P0**: Logit Lens — prediction crystallizes at layer 6
- **P2**: Linear Probes — decoding semantic concepts from activations
- **P3**: MLP Translator — non-linear concept decoding
- **P4**: Activation Steering — testing causality

**Runtime**: ~30 minutes on CPU (macOS/Windows/Linux)


In [ ]:
# Install dependencies (uncomment if needed)
# !pip install transformer-lens torch numpy matplotlib seaborn scikit-learn spacy tqdm
# !python -m spacy download en_core_web_sm

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from transformer_lens import HookedTransformer

DEVICE = 'cpu'
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

## P0 — Logit Lens: Where Does the Prediction Form?

In [ ]:
# Load GPT-2 small
model = HookedTransformer.from_pretrained('gpt2', device=DEVICE)
text = 'The cat sat on the mat and looked'
tokens = model.to_tokens(text)
logits, cache = model.run_with_cache(tokens)

# Extract residual stream at layer 6 (index 5)
resid = cache['blocks.5.hook_resid_post']
layer_logits = resid @ model.W_U

# Top 5 predictions after "looked"
probs = torch.softmax(layer_logits[0, -1], dim=-1)
top5 = torch.topk(probs, 5)

print(f"Layer 6 predictions after 'looked':")
for i, (p, idx) in enumerate(zip(top5.values, top5.indices)):
    word = model.to_single_str_token(idx.item())
    print(f"{i+1}. '{word}' : {p.item():.4f} ({p.item()*100:.2f}%)")

In [ ]:
# Rank percentile evolution across layers
true_token_id = tokens[0, -1].item()
ranks = []

for layer in range(12):
    resid = cache[f'blocks.{layer}.hook_resid_post']
    layer_logits = resid @ model.W_U
    
    # Rank of true token
    sorted_indices = torch.argsort(layer_logits[0, -1], descending=True)
    rank = (sorted_indices == true_token_id).nonzero().item()
    rank_percentile = rank / model.cfg.d_vocab * 100
    ranks.append(rank_percentile)

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(12), ranks, marker='o', linewidth=2, markersize=8)
ax.axhline(5, color='green', linestyle='--', alpha=0.5, label='Top 5% threshold')
ax.set_xlabel('Layer')
ax.set_ylabel('Rank Percentile (%)')
ax.set_title(f"Rank Percentile of True Token: '{model.to_single_str_token(true_token_id)}'")
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0, max(ranks) * 1.1)
plt.tight_layout()
plt.show()

print(f"\nInput rank: {ranks[0]:.1f}%")
print(f"Layer 6 rank: {ranks[5]:.1f}%")
print(f"Improvement: {ranks[0]/ranks[5]:.1f}x")

## P2 — Linear Probes: Reading Concepts from Activations

In [ ]:
import pickle
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report

# Load pre-computed data (from P1)
DATA_DIR = Path('data')
activations_dict = torch.load(DATA_DIR / 'activations.pt', weights_only=False)
labels_dict = np.load(DATA_DIR / 'labels.npy', allow_pickle=True).item()

BEST_LAYER = 6
X = activations_dict[BEST_LAYER]  # [N, 768]

print(f"Loaded activations: {X.shape}")
print(f"Concepts: {list(labels_dict.keys())}")

In [ ]:
# Train probes for each concept
results = {}

for concept in labels_dict.keys():
    y = labels_dict[concept]
    
    # Skip if too few positives
    if y.sum() < 5:
        print(f"Skip {concept}: only {y.sum()} positives")
        continue
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    
    probe = LogisticRegression(max_iter=1000, class_weight='balanced')
    probe.fit(X_train, y_train)
    
    y_pred = probe.predict(X_test)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    results[concept] = f1

# Display results
print(f"\n{'Concept':<12} | {'F1':<6}")
print("-" * 25)
for concept, f1 in sorted(results.items(), key=lambda x: x[1], reverse=True):
    print(f"{concept:<12} | {f1:<6.3f}")

macro_f1 = np.mean(list(results.values()))
print(f"\nMacro F1: {macro_f1:.3f}")

## P3 — MLP vs Linear: Does Non-Linearity Help?

In [ ]:
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

class ConceptTranslator(nn.Module):
    def __init__(self, d_model=768, hidden=128, n_concepts=10):
        super().__init__()
        self.encoder = nn.Linear(d_model, hidden)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)
        self.decoder = nn.Linear(hidden, n_concepts)
    
    def forward(self, x):
        h = self.relu(self.encoder(x))
        h = self.dropout(h)
        return self.decoder(h)

# Build multi-label dataset
y_all = np.stack([labels_dict[c] for c in labels_dict.keys()], axis=1)  # [N, 10]
X_train, X_test, y_train, y_test = train_test_split(
    X, y_all, test_size=0.2, random_state=42
)

# Convert to tensors
X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32)

# DataLoader
train_ds = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)

# Model
mlp = ConceptTranslator(d_model=768, hidden=128, n_concepts=10)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(mlp.parameters(), lr=1e-3)

# Train
mlp.train()
for epoch in range(20):
    total_loss = 0
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        logits = mlp(batch_x)
        loss = criterion(logits, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    if epoch % 5 == 0:
        print(f"Epoch {epoch}: loss={total_loss/len(train_loader):.4f}")

print("Training complete!")

In [ ]:
# Evaluate MLP
mlp.eval()
with torch.no_grad():
    test_logits = mlp(X_test_t)
    test_probs = torch.sigmoid(test_logits)

# Per-concept F1
mlp_f1s = {}
for i, concept in enumerate(labels_dict.keys()):
    y_true = y_test[:, i]
    y_pred = (test_probs[:, i] > 0.5).numpy().astype(int)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    mlp_f1s[concept] = f1

# Compare Linear vs MLP
print(f"{'Concept':<12} | {'Linear':<8} | {'MLP':<8} | {'Diff':<8}")
print("-" * 45)
for concept in labels_dict.keys():
    lin = results.get(concept, 0)
    mlp = mlp_f1s.get(concept, 0)
    diff = mlp - lin
    print(f"{concept:<12} | {lin:<8.3f} | {mlp:<8.3f} | {diff:+<8.3f}")

print(f"\nLinear Macro F1: {np.mean(list(results.values())):.3f}")
print(f"MLP Macro F1: {np.mean(list(mlp_f1s.values())):.3f}")

## P4 — Steering: Testing Causality

In [ ]:
# Load pre-computed steering results
with open(DATA_DIR / 'p4_steering_results.pkl', 'rb') as f:
    steering_results = pickle.load(f)

# Plot
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

concepts = ['DATE', 'NUMBER', 'NOUN', 'PROPN', 'VERB', 'ORG']

for idx, concept in enumerate(concepts):
    ax = axes[idx]
    if concept not in steering_results:
        continue
    
    for prompt, probs in steering_results[concept].items():
        if not probs:
            continue
        alphas = sorted(probs.keys())
        values = [probs[a] for a in alphas]
        label = prompt[:20] + '...' if len(prompt) > 20 else prompt
        ax.plot(alphas, values, marker='o', label=label)
    
    ax.axvline(0, color='gray', linestyle='--', alpha=0.5)
    ax.set_xlabel('Steering coefficient')
    ax.set_ylabel('P(target token)')
    ax.set_title(f'Concept: {concept}')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.suptitle('P4 — Activation Steering Results', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nVerdict: Flat curves = directions are correlational, NOT causal.")

## Summary

| Phase | Result | Status |
|-------|--------|--------|
| P0 | Decision layer = Layer 6 (not 12) | Complete |
| P2 | Linear probes: F1=0.800 @ L6 | Complete |
| P3 | MLP: F1=0.814 (+1.4% vs linear) | Complete |
| P4 | Steering: NOT causal (mean-diff) | Complete (negative result) |

**Next steps**: Scale dataset to 50k tokens, test on larger models, use adversarial contrast pairs for real steering.